# BiMba: Example Inference Notebook
This notebook demonstrates how to load the trained BiMba model, score the residues of a protein as interface or non-interface, and compute the following evaluation metrics for a single protein example: 

- **Precision**
- **Recall**
- **F1-score**
- **Accuracy**
- **AUC (ROC AUC)**
- **Average Precision (PRC)**


### Testing on two protein examples

#### Part 1: Reading the trained model


In [1]:
import os
import torch
from utils.dataset import PDB_complex_testing
from network.bimba_vim import get_ml_config
from bimba_train import Bsite_proto


MODEL_DIR='./model/'

MODEL_NAME='bimba_trained_model'
def load_model(model_dir, model_name, device):
    params = {
        "dim_head": 16,
        "hidden_size": 128,
        "dropout": 0.0,
        "patch_size": 3,
        "depth": 8,
    }

    model_config = get_ml_config(params)
    model = Bsite_proto(model_config, img_size=18, num_classes=2).float().to(device)
    state_dict = torch.load(os.path.join(model_dir, f"{model_name}.pth"), map_location=device)
    model.load_state_dict(state_dict)
    model.eval()
    return model

/aul/homes/ashir018/.conda/envs/vim-site/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-11-21 13:05:55,365	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2025-11-21 13:05:55,512	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


Data directory set to: /a/bear.cs.fiu.edu./disk/bear-b/users/ashir018/PIsToN_Interface/BiMba/test_example/data_preparation/


#### Part 2: Preparing Test dataset

In [2]:

data_prepare_dir = "./test_example/data_preparation"
grid_dir = os.path.join(data_prepare_dir, "07-grid")
feat_dir = os.path.join(data_prepare_dir, "external_feats")
grid_info_dir = os.path.join(data_prepare_dir, "08-patch_info")
pdb_list = os.path.join(data_prepare_dir, "../test_list.txt")
pdbs = os.path.join(data_prepare_dir, "04-chains_pdbs")

ground_truth = os.path.join(data_prepare_dir, "../ground_truth")
ground_truth_ply = os.path.join(data_prepare_dir, "05-surface_ply")

testing_pdbs = set(x.strip() for x in open(pdb_list).readlines())
output_results_dir = os.path.join(os.getcwd(), "test_example", "Test_Results")
os.makedirs(output_results_dir, exist_ok=True)
pred_dir = output_results_dir

In [3]:
def load_ppi_list():
    with open(pdb_list, 'r') as f:
        ppi_list = [line.strip() for line in f if line.strip()]
    print("ppi_list : ", ppi_list)
    return ppi_list

#### Part 3: Obtaining BiMba Scores 

In [4]:
import torch
import numpy as np
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, average_precision_score, accuracy_score
import pandas as pd
import meshio
import warnings
import numpy as np

In [5]:
import csv
from torch.utils.data import DataLoader

def run_inference(model, test_loader, device):
    feature_subset = list(range(5))
    model = model.to(device).eval()
    scores_dir = os.path.join(output_results_dir, "scores")
    os.makedirs(scores_dir, exist_ok=True)

    for ppi in ppi_list:
        csv_file = os.path.join(scores_dir, f"{ppi}.csv")
        if os.path.exists(csv_file):
            continue
        test_db = PDB_complex_testing(
            ppi_list=[ppi],
            data_prepare_dir=data_prepare_dir,
            device=device,
            feature_subset=feature_subset
        )
        all_patches_dict = test_db.all_patches_dict  # {res_id: [grid_ids]}
        print("all_patches_dict for PPI {} is: ".format(ppi), all_patches_dict)
        if len(test_db) == 0:
            print(f"No valid patches found for {ppi}, skipping.")
            continue
        test_loader = DataLoader(
            test_db,
            batch_size=1,
            shuffle=False,
            pin_memory=False
        )

        patch_info = []
        for res_id, grid_ids in all_patches_dict.items():
            res_num = res_id[1]
            for grid_id in grid_ids:
                patch_info.append((res_num, grid_id))

        # Run inference and collect scores
        raw_probs = []
        with torch.no_grad():
            for i, (grid, _, extra_feats) in enumerate(test_loader):
                grid = grid[:, feature_subset, :, :].to(device)
                extra_feats = extra_feats.to(device)
                logits, _ = model(grid, extra_feats)
                prob = torch.sigmoid(logits).item()
                raw_probs.append(prob)

        # Save to CSV: res_num, grid_id, score
        with open(csv_file, mode='w', newline='') as file:
            writer = csv.writer(file)
            writer.writerow(["res_id", "grid_id", "Score"])
            for (res_id, grid_id), score in zip(patch_info, raw_probs):
                writer.writerow([res_id, grid_id, score])
    return

In [6]:
def labels_per_residue(myid):
    label_file = os.path.join(ground_truth, f"{myid}.txt")
    labels_per_res = {}
    with open(label_file, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) >= 4:
                res_id = int(parts[1])
                label = int(parts[3])
                labels_per_res[res_id] = label
    return labels_per_res

In [7]:
from Bio.PDB import PDBParser, Selection
def pred_labels_per_residue_scores(myid, res_ids_scores):
    raw_labels = labels_per_residue(myid)
    residue_labels = {(" ", res_id, " "): label for res_id, label in raw_labels.items()}
    parser = PDBParser()
    pdb_path = os.path.join(pdbs, f"{myid}.pdb")
    struct = parser.get_structure(pdb_path, pdb_path)
    _ = Selection.unfold_entities(struct, "R")
    pred_dict = dict(res_ids_scores)
    pred_scores, true_labels = [], []
    for res_id, true_label in residue_labels.items():
        res_num = res_id[1]
        pred_scores.append(pred_dict.get(res_num, 0.0))
        true_labels.append(true_label)
    return pred_scores, true_labels

In [8]:
def bimba_scores(scores_dir):
    warnings.filterwarnings("ignore")

    # residue accumulators
    point_scores = []
    point_labels = []
    auc_per_protein = {}
    protein_names = []
    labels_per_res = []
    preds_per_res = []
    auc_per_res = []

    pred_results = {}

    for filename in os.listdir(scores_dir):
        if not filename.endswith(".csv"):
            continue
        myid = filename[:-4]
        if myid not in testing_pdbs:
            continue

        pred_df = pd.read_csv(os.path.join(scores_dir, filename))
        pred = pred_df["Score"].values.astype(np.float64)
        res_id = pred_df["res_id"].values
        grid_ids = pred_df["grid_id"].values

        # ---- per-point labels ----
        mesh_file = os.path.join(ground_truth_ply, f"{myid}.ply")
        if not os.path.exists(mesh_file):
            continue
        mesh = meshio.read(mesh_file)
        iface = mesh.point_data["iface"]
        labels_this_protein = iface[grid_ids]
        point_scores.append(pred)
        point_labels.append(labels_this_protein)

        # Per-point AUC
        try:
            auc_val = roc_auc_score(labels_this_protein, pred)
        except ValueError:
            auc_val = np.nan
        auc_per_protein[myid] = auc_val

        # ---- per-residue scores (max pooling) ----
        res_scores = []
        for rid in np.unique(res_id):
            max_score = float(np.max(pred[res_id == rid]))
            res_scores.append((int(rid), max_score))

        # ---- per-residue true labels ----
        label_file = os.path.join(ground_truth, f"{myid}.txt")
        if not os.path.exists(label_file):
            continue
        preds_res, labels_res = pred_labels_per_residue_scores(myid, res_scores)
        pred_results[myid] = (preds_res, labels_res)

        # Per-residue AUC
        try:
            this_auc_res = roc_auc_score(labels_res, preds_res)
        except ValueError:
            this_auc_res = np.nan

        # Append to residue lists
        auc_per_res.append(this_auc_res)
        preds_per_res.append(preds_res)
        labels_per_res.append(labels_res)
        protein_names.append(myid)

    return (
        point_scores,
        point_labels,
        auc_per_protein,
        protein_names,
        labels_per_res,
        preds_per_res,
        auc_per_res,
        pred_results
    )

#### Part 3: Classification Metrics

In [9]:
def setup_environment():
    torch.cuda.set_device(3)
    if not torch.cuda.is_available():
        raise RuntimeError("No GPU found! CUDA is required.")
    return torch.device("cuda")

In [10]:
def eval_metrics(probs, targets, cal_curves=True):
    """
    Compute evaluation metrics for binary classification:
    - ROC AUC
    - PRC (Average Precision)
    - Precision
    - Recall
    - F1 score
    - Accuracy
    """
    probs = np.asarray(probs, dtype=np.float64)
    targets = np.asarray(targets, dtype=np.int32)
    auc_val, auprc_val = np.nan, np.nan
    if cal_curves and len(np.unique(targets)) > 1:
        auc_val = roc_auc_score(targets, probs)
        auprc_val = average_precision_score(targets, probs)
    pred_bin = (probs > 0.5).astype(int)

    precision = precision_score(targets, pred_bin, zero_division=0)
    recall = recall_score(targets, pred_bin, zero_division=0)
    f1 = f1_score(targets, pred_bin, zero_division=0)
    accuracy = accuracy_score(targets, pred_bin)

    print("AUC, PRC, Precision, Recall, F1, Accuracy are: ",
          auc_val, auprc_val, precision, recall, f1, accuracy)
    return auc_val, auprc_val, precision, recall, f1, accuracy

In [11]:
device = setup_environment()
model = load_model(MODEL_DIR, MODEL_NAME, device)
ppi_list = load_ppi_list()
run_inference(model, ppi_list, device)

scores_dir = os.path.join(output_results_dir, "scores")
(
    bimba_scores,
    labels,
    auc_per_protein,
    names,
    labels_per_res,
    pred_per_res,
    auc_per_res,
    pred_results
) = bimba_scores(scores_dir)

# ---------- Per-protein metrics ----------
per_protein_results = []
for protein_id, (preds, labels) in pred_results.items():
    print("Processing protein:", protein_id)
    preds = np.asarray(preds, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.int32)
    if preds.size == 0 or labels.size == 0:
        continue

    auroc, auprc, pre, rec, f1, acc = eval_metrics(preds, labels, cal_curves=True)
    per_protein_results.append({
        "Protein": protein_id,
        "Recall": rec,
        "Precision": pre,
        "F1": f1,
        "Accuracy": acc,
        "AUROC": auroc,
        "AUPRC": auprc
    })

per_protein_df = pd.DataFrame(per_protein_results)
per_protein_df.to_csv(os.path.join(output_results_dir, "Metrics_results_per_protein.csv"), index=False)
print("✅ Per-protein metrics saved to Metrics_results_per_protein.csv")
print(per_protein_df)

ppi_list :  ['6GR8_B', '6C7Y_B']
number of residues for PPI 6GR8_B is : 64
Fallback to other atom: HA
Fallback to other atom: HD22
Fallback to other atom: HZ1
Fallback to other atom: HZ1
Fallback to other atom: HG
Fallback to other atom: HB
Fallback to other atom: HG23
selected 425 number of points instead of 2379
all_patches_dict for PPI 6GR8_B is:  {(' ', 1, ' '): [71, 302, 313, 316, 318, 428, 962, 1241, 1523, 1573, 2101], (' ', 2, ' '): [528, 1147, 1255, 2011, 221, 13, 30, 162, 458, 657, 1270, 2106], (' ', 3, ' '): [571, 640, 1017, 1065, 1096, 1111, 1429, 1735], (' ', 4, ' '): [122, 256, 398, 609, 1313, 1439, 1629], (' ', 5, ' '): [77, 631, 1686], (' ', 6, ' '): [1013], (' ', 7, ' '): [1073, 2122, 2349, 1023, 1707], (' ', 8, ' '): [932, 78, 457, 1106, 1394, 1493, 1503, 1562, 2229, 915, 1095, 2184], (' ', 9, ' '): [923, 1325, 1346, 2017], (' ', 10, ' '): [338, 1472, 1580], (' ', 11, ' '): [1608, 1748, 1913], (' ', 12, ' '): [720, 1266, 1309], (' ', 13, ' '): [1992, 2265], (' ', 14, '